# Etapa 7

Nesta etapa, o objetivo é transformar as análises realizadas nas etapas anteriores em decisões estratégicas para a empresa Comercial Insight Ltda.

Enquanto as etapas anteriores tiveram foco em entendimento do problema, diagnóstico, tratamento, análise estatística e modelagem para BI, esta etapa possui foco executivo. Ou seja, o objetivo principal não é apenas descrever os dados, mas interpretar os resultados e indicar quais decisões a empresa deve tomar com base nas evidências encontradas.

A Etapa 7 será desenvolvida antes da Etapa 6 porque os insights e recomendações estratégicas servirão como base para definir quais visualizações deverão compor o dashboard final. Dessa forma, a visualização será construída para comunicar os principais achados de negócio, e não apenas para exibir gráficos isolados.

In [ ]:
# ETAPA 7 - INSIGHTS E TOMADA DE DECISÃO
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
# CARREGAMENTO DA BASE TRATADA
# ============================================================

caminho_base = Path("dados_empresa_tratado.csv")

if not caminho_base.exists():
    raise FileNotFoundError(
        "O arquivo 'dados_empresa_tratado.csv' não foi encontrado. "
    )

df = pd.read_csv(caminho_base)

df["data"] = pd.to_datetime(df["data"], errors="coerce")

df.head()

# ============================================================
# VALIDAÇÃO DA BASE
# ============================================================

validacao_base = pd.DataFrame({
    "Métrica": [
        "Quantidade de registros",
        "Quantidade de colunas",
        "Valores nulos",
        "Linhas duplicadas",
        "Datas inválidas",
        "IDs únicos de venda"
    ],
    "Resultado": [
        df.shape[0],
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum(),
        df["data"].isna().sum(),
        df["id_venda"].nunique()
    ]
})

validacao_base

,Métrica,Resultado
0,Quantidade de registros,1000
1,Quantidade de colunas,27
2,Valores nulos,0
3,Linhas duplicadas,0
4,Datas inválidas,0
5,IDs únicos de venda,1000


In [ ]:
# CRIAÇÃO DE MÉTRICAS AUXILIARES
# ============================================================

# Transformar atingimento de meta em variável numérica
df["atingiu_meta_flag"] = np.where(df["atingiu_meta"] == "Sim", 1, 0)

# Criar margem de lucro por venda
df["margem_lucro_venda"] = np.where(
    df["receita"] != 0,
    df["lucro"] / df["receita"],
    np.nan
)

# Criar faixas de desconto para análise executiva
df["faixa_desconto"] = pd.cut(
    df["desconto"],
    bins=[-0.001, 0.05, 0.10, 0.15, 0.20, 1],
    labels=[
        "0% a 5%",
        "5% a 10%",
        "10% a 15%",
        "15% a 20%",
        "Acima de 20%"
    ]
)

df[["receita", "lucro", "margem_lucro_venda", "desconto", "faixa_desconto", "atingiu_meta", "atingiu_meta_flag"]]

,receita,lucro,margem_lucro_venda,desconto,faixa_desconto,atingiu_meta,atingiu_meta_flag
0,"19,258.19","3,396.89",0.18,0.18,15% a 20%,Não,0
1,"4,457.55","1,380.95",0.31,0.12,10% a 15%,Não,0
2,362.16,72.78,0.20,0.23,Acima de 20%,Não,0
3,"21,446.22","5,104.80",0.24,0.21,Acima de 20%,Não,0
4,"21,105.90",758.70,0.04,0.17,15% a 20%,Não,0
...,...,...,...,...,...,...,...
995,"2,707.37",-133.93,-0.05,0.15,10% a 15%,Não,0
996,"3,538.86","1,349.11",0.38,0.11,10% a 15%,Não,0
997,"2,049.59",132.29,0.06,0.04,0% a 5%,Sim,1
998,"10,151.55","3,324.84",0.33,0.24,Acima de 20%,Não,0


In [ ]:
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_percentual(valor):
    return f"{valor:.2%}".replace(".", ",")


def analisar_dimensao(base, dimensao):
    """
    Gera análise executiva por dimensão de negócio.
    Calcula vendas, receita, lucro, margem, desconto médio,
    taxa de meta e taxa de atraso.
    """
    analise = base.groupby(dimensao, observed=True).agg(
        vendas=("id_venda", "count"),
        receita_total=("receita", "sum"),
        lucro_total=("lucro", "sum"),
        desconto_medio=("desconto", "mean"),
        taxa_atingiu_meta=("atingiu_meta_flag", "mean"),
        taxa_atraso=("status_entrega", lambda x: (x == "Atrasado").mean())
    ).reset_index()

    analise["margem_lucro"] = analise["lucro_total"] / analise["receita_total"]
    analise["ticket_medio"] = analise["receita_total"] / analise["vendas"]

    return analise.sort_values("receita_total", ascending=False)

In [ ]:
# KPIs FINAIS
# ============================================================

receita_total = df["receita"].sum()
lucro_total = df["lucro"].sum()
margem_lucro = lucro_total / receita_total
quantidade_total = df["quantidade"].sum()
ticket_medio = df["receita"].mean()
meta_total = df["meta_venda"].sum()
taxa_cumprimento_meta_total = receita_total / meta_total
taxa_meta = (df["atingiu_meta"] == "Sim").mean()
taxa_atraso = (df["status_entrega"] == "Atrasado").mean()

kpis_finais = pd.DataFrame({
    "Indicador": [
        "Receita Total",
        "Lucro Total",
        "Margem de Lucro",
        "Quantidade Vendida",
        "Ticket Médio",
        "Meta Total",
        "Cumprimento da Meta Total",
        "Taxa de Vendas que Atingiram Meta",
        "Taxa de Entregas Atrasadas"
    ],
    "Valor": [
        formatar_moeda(receita_total),
        formatar_moeda(lucro_total),
        formatar_percentual(margem_lucro),
        f"{quantidade_total:,}".replace(",", "."),
        formatar_moeda(ticket_medio),
        formatar_moeda(meta_total),
        formatar_percentual(taxa_cumprimento_meta_total),
        formatar_percentual(taxa_meta),
        formatar_percentual(taxa_atraso)
    ]
})

kpis_finais

,Indicador,Valor
0,Receita Total,"R$ 15.196.118,48"
1,Lucro Total,"R$ 3.081.804,68"
2,Margem de Lucro,"20,28%"
3,Quantidade Vendida,6.062
4,Ticket Médio,"R$ 15.196,12"
5,Meta Total,"R$ 18.353.644,52"
6,Cumprimento da Meta Total,"82,80%"
7,Taxa de Vendas que Atingiram Meta,"7,00%"
8,Taxa de Entregas Atrasadas,"51,90%"


### Observação
A empresa apresenta um volume relevante de vendas, com receita total aproximada de R$ 15,20 milhões e lucro total de aproximadamente R$ 3,08 milhões. A margem geral ficou em torno de 20,28%, indicando que a operação é lucrativa de forma agregada.

Entretanto, dois indicadores chamam atenção: apenas 7,00% das vendas atingiram a meta individual e 51,90% das entregas foram classificadas como atrasadas. Isso mostra que, embora a empresa gere receita e lucro, existem problemas relevantes de desempenho comercial e eficiência operacional.

## Análise por Dimensões

In [ ]:
# ANÁLISE EXECUTIVA POR REGIÃO
# ============================================================

analise_regiao = analisar_dimensao(df, "regiao")

analise_regiao

,regiao,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
4,Sul,209,"3,314,346.57","732,603.31",0.12,0.05,0.55,0.22,"15,858.12"
0,Centro-Oeste,202,"3,265,071.81","586,749.50",0.14,0.05,0.49,0.18,"16,163.72"
2,Norte,209,"3,247,788.94","689,213.02",0.12,0.11,0.52,0.21,"15,539.66"
3,Sudeste,199,"2,720,463.90","527,492.37",0.13,0.07,0.53,0.19,"13,670.67"
1,Nordeste,181,"2,648,447.26","545,746.48",0.13,0.08,0.50,0.21,"14,632.31"


### Dimensão Região
A análise por região mostra diferenças importantes de desempenho. A região Sul apresentou a maior combinação de receita, lucro e margem, tornando-se a principal referência de desempenho regional.

Por outro lado, o Centro-Oeste apresentou uma das maiores receitas, mas a menor margem de lucro. Isso indica que a região vende bem, mas transforma menos receita em lucro. Esse padrão sugere necessidade de investigar descontos, custos, mix de produtos e atuação comercial nessa região.

In [ ]:
# ANÁLISE EXECUTIVA POR PRODUTO
# ============================================================

analise_produto = analisar_dimensao(df, "produto")

analise_produto

,produto,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
2,Monitor,143,"2,463,025.32","448,537.86",0.13,0.10,0.52,0.18,"17,223.95"
7,Teclado,115,"1,973,461.44","425,008.67",0.13,0.05,0.57,0.22,"17,160.53"
3,Mouse,133,"1,962,335.25","389,571.59",0.12,0.08,0.53,0.20,"14,754.40"
5,Smartphone,133,"1,946,475.02","402,535.95",0.13,0.04,0.49,0.21,"14,635.15"
1,Impressora,115,"1,859,469.92","419,170.28",0.13,0.06,0.51,0.23,"16,169.30"
0,Headset,111,"1,697,277.71","310,612.35",0.14,0.05,0.54,0.18,"15,290.79"
4,Notebook,122,"1,647,800.91","386,693.22",0.12,0.08,0.52,0.23,"13,506.56"
6,Tablet,128,"1,646,272.91","299,674.76",0.13,0.08,0.46,0.18,"12,861.51"


### Dimensão Produto
A análise por produto permite observar que os produtos com maior receita não necessariamente são os produtos mais rentáveis proporcionalmente.

O Monitor apresentou a maior receita total, mas sua margem ficou abaixo de produtos como Notebook e Impressora. Isso indica que decisões comerciais não devem ser baseadas apenas em faturamento, mas também em lucro e margem.

In [ ]:
#  ANÁLISE EXECUTIVA POR CATEGORIA
# ============================================================

analise_categoria = analisar_dimensao(df, "categoria")

analise_categoria

,categoria,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
0,Informática,492,"7,297,489.88","1,502,208.42",0.13,0.08,0.50,0.21,"14,832.30"
2,Periféricos,346,"5,432,341.12","1,091,976.27",0.13,0.06,0.55,0.20,"15,700.41"
3,Telefonia,132,"1,934,879.54","399,791.57",0.13,0.04,0.49,0.21,"14,658.18"
1,Outros,30,"531,407.95","87,828.43",0.12,0.10,0.53,0.17,"17,713.60"


### Dimensão Categoria
A categoria Informática é a principal fonte de receita e lucro da empresa, seguida por Periféricos. Essas duas categorias concentram grande parte do resultado financeiro e devem receber prioridade em campanhas, estoque e estratégia comercial.

A categoria Outros possui menor representatividade e menor margem, indicando que pode exigir revisão de posicionamento, preço ou mix de produtos.

In [ ]:
# ANÁLISE EXECUTIVA POR CANAL DE VENDA
# ============================================================

analise_canal = analisar_dimensao(df, "canal_venda")

analise_canal

,canal_venda,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
1,Online,522,"8,092,223.75","1,600,532.68",0.13,0.07,0.49,0.20,"15,502.34"
0,Loja,478,"7,103,894.73","1,481,272.00",0.13,0.06,0.55,0.21,"14,861.70"


### Dimensão Venda
O canal Online apresentou maior receita e maior lucro absoluto. Isso indica maior força de escala e maior participação no resultado total da empresa.

A Loja apresentou margem de lucro ligeiramente superior, mas também maior taxa de atraso. Portanto, o canal físico parece ser proporcionalmente rentável, mas possui maior fragilidade operacional.

In [ ]:
# ANÁLISE EXECUTIVA POR VENDEDOR
# ============================================================

analise_vendedor = analisar_dimensao(df, "vendedor")

# Ordenar por lucro total para avaliação executiva
analise_vendedor = analise_vendedor.sort_values("lucro_total", ascending=False)

analise_vendedor

,vendedor,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
19,Vendedor_20,49,"905,934.02","239,532.82",0.13,0.06,0.55,0.26,"18,488.45"
12,Vendedor_13,60,"1,050,462.62","206,493.86",0.13,0.07,0.50,0.20,"17,507.71"
17,Vendedor_18,53,"958,083.42","203,100.38",0.12,0.08,0.36,0.21,"18,077.05"
3,Vendedor_04,55,"783,594.95","193,736.34",0.11,0.13,0.40,0.25,"14,247.18"
18,Vendedor_19,50,"908,888.43","189,655.63",0.15,0.06,0.50,0.21,"18,177.77"
9,Vendedor_10,63,"832,485.63","187,059.08",0.13,0.03,0.62,0.22,"13,214.06"
5,Vendedor_06,50,"733,731.74","165,302.81",0.12,0.02,0.56,0.23,"14,674.63"
7,Vendedor_08,46,"809,109.51","161,494.04",0.14,0.02,0.63,0.20,"17,589.34"
1,Vendedor_02,48,"723,128.05","159,101.63",0.11,0.17,0.56,0.22,"15,065.17"
16,Vendedor_17,50,"800,704.41","157,393.44",0.12,0.14,0.46,0.20,"16,014.09"


### Dimensão Vendedor
A análise por vendedor mostra que a avaliação comercial não deve considerar apenas receita. Alguns vendedores apresentam receita elevada, mas margem menor, enquanto outros se destacam pelo lucro e pela rentabilidade.

O vendedor com maior lucro total foi o Vendedor_20, enquanto o Vendedor_13 liderou em receita. Essa diferença reforça que a empresa deve avaliar a equipe comercial por múltiplos critérios: receita, lucro, margem, desconto médio e atingimento de meta.

In [ ]:
# ANÁLISE EXECUTIVA POR TIPO DE CLIENTE
# ============================================================

analise_tipo_cliente = analisar_dimensao(df, "tipo_cliente")

analise_tipo_cliente

,tipo_cliente,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
1,Recorrente,519,"7,853,078.63","1,578,620.54",0.13,0.08,0.49,0.20,"15,131.17"
0,Novo,481,"7,343,039.85","1,503,184.14",0.13,0.06,0.56,0.20,"15,266.20"


### Dimensão Cliente
Clientes recorrentes geraram maior receita e maior lucro absoluto, o que reforça a importância da fidelização e do relacionamento com clientes já existentes.

Clientes novos, por outro lado, apresentaram margem ligeiramente superior. Isso indica que a empresa também deve manter estratégias de aquisição, desde que os descontos sejam controlados.

### Análise de Descontos
A análise dos descontos revelou uma relação negativa entre desconto e margem de lucro. Em termos práticos, quanto maior o desconto, menor tende a ser a margem da venda.

A análise por faixa de desconto reforça essa conclusão: vendas com descontos baixos apresentam margem significativamente maior, enquanto vendas com desconto acima de 20% possuem margem bastante reduzida.

Essa é uma das principais evidências estratégicas do projeto, pois indica que a empresa pode estar sacrificando rentabilidade para gerar vendas.

In [ ]:
# ANÁLISE EXECUTIVA POR FAIXA DE DESCONTO
# ============================================================

analise_desconto = analisar_dimensao(df, "faixa_desconto")

# Ordenar na sequência correta das faixas
ordem_faixas = [
    "0% a 5%",
    "5% a 10%",
    "10% a 15%",
    "15% a 20%",
    "Acima de 20%"
]

analise_desconto["faixa_desconto"] = pd.Categorical(
    analise_desconto["faixa_desconto"],
    categories=ordem_faixas,
    ordered=True
)

analise_desconto = analise_desconto.sort_values("faixa_desconto")

analise_desconto

,faixa_desconto,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
0,0% a 5%,202,"3,360,463.69","1,007,026.62",0.03,0.28,0.51,0.30,"16,635.96"
1,5% a 10%,183,"2,729,813.81","664,347.28",0.08,0.07,0.45,0.24,"14,917.02"
2,10% a 15%,218,"3,435,365.48","667,703.57",0.13,0.00,0.56,0.19,"15,758.56"
3,15% a 20%,219,"3,335,380.14","528,110.90",0.18,0.00,0.51,0.16,"15,230.05"
4,Acima de 20%,178,"2,335,095.36","214,616.31",0.23,0.00,0.56,0.09,"13,118.51"


In [ ]:
# CORRELAÇÃO ENTRE DESCONTO E MARGEM DE LUCRO POR VENDA
# ============================================================

correlacao_pearson = df["desconto"].corr(df["margem_lucro_venda"], method="pearson")
correlacao_spearman = df["desconto"].corr(df["margem_lucro_venda"], method="spearman")

analise_correlacao_desconto = pd.DataFrame({
    "Método": ["Pearson", "Spearman"],
    "Correlação entre desconto e margem": [
        correlacao_pearson,
        correlacao_spearman
    ]
})

analise_correlacao_desconto

,Método,Correlação entre desconto e margem
0,Pearson,-0.45
1,Spearman,-0.43


### Análise de Entrega
A taxa de entregas atrasadas é elevada e deve ser tratada como um problema estratégico. As vendas entregues no prazo apresentaram maior taxa de atingimento de meta do que as vendas atrasadas.

Embora essa análise não prove causalidade direta, ela indica uma associação importante entre eficiência operacional e desempenho comercial. Portanto, a entrega deve ser acompanhada como KPI de gestão, e não apenas como uma informação operacional.

In [ ]:
# ANÁLISE EXECUTIVA POR STATUS DE ENTREGA
# ============================================================

analise_entrega = df.groupby("status_entrega").agg(
    vendas=("id_venda", "count"),
    receita_media=("receita", "mean"),
    lucro_medio=("lucro", "mean"),
    taxa_atingiu_meta=("atingiu_meta_flag", "mean"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

analise_entrega["margem_media_aproximada"] = (
    analise_entrega["lucro_medio"] / analise_entrega["receita_media"]
)

analise_entrega

,status_entrega,vendas,receita_media,lucro_medio,taxa_atingiu_meta,receita_total,lucro_total,margem_media_aproximada
0,Atrasado,519,"15,112.40","3,111.54",0.06,"7,843,337.26","1,614,887.26",0.21
1,No prazo,481,"15,286.45","3,049.72",0.08,"7,352,781.22","1,466,917.42",0.20


### Análise de Marca e Linhas de produto
A marca Alpha apresentou a maior contribuição para o lucro total e também uma das melhores margens, indicando força comercial e financeira.

As linhas Econômico, Premium e Padrão apresentaram contribuição relativamente equilibrada para o lucro. Isso mostra que a empresa não depende de apenas um padrão de produto para gerar resultado, mas deve acompanhar a rentabilidade de cada linha separadamente.

In [ ]:
# ANÁLISE EXECUTIVA POR MARCA
# ============================================================

analise_marca = analisar_dimensao(df, "marca")

analise_marca

,marca,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
0,Alpha,255,"4,304,476.31","944,584.34",0.12,0.07,0.49,0.22,"16,880.30"
3,Gamma,252,"3,811,106.85","726,318.26",0.13,0.09,0.52,0.19,"15,123.44"
1,Beta,250,"3,560,376.89","731,897.15",0.13,0.05,0.54,0.21,"14,241.51"
2,Delta,243,"3,520,158.43","679,004.93",0.13,0.07,0.53,0.19,"14,486.25"


In [ ]:
# ANÁLISE EXECUTIVA POR LINHA DE PRODUTO
# ============================================================

analise_linha = analisar_dimensao(df, "linha_produto")

analise_linha

,linha_produto,vendas,receita_total,lucro_total,desconto_medio,taxa_atingiu_meta,taxa_atraso,margem_lucro,ticket_medio
1,Padrão,323,"5,183,979.29","1,004,272.99",0.13,0.07,0.53,0.19,"16,049.47"
0,Econômico,347,"5,084,052.45","1,052,581.71",0.12,0.07,0.53,0.21,"14,651.45"
2,Premium,330,"4,928,086.74","1,024,949.98",0.13,0.07,0.50,0.21,"14,933.60"


### Analise de Pareto
A análise de Pareto mostrou que a concentração mais clara ocorre nas categorias de produto. As categorias Informática e Periféricos concentram mais de 80% do lucro total da empresa.

Já em produtos, regiões, marcas, linhas e vendedores, o lucro está mais distribuído. Isso significa que a empresa não depende exclusivamente de um único produto, região ou vendedor, mas possui algumas categorias prioritárias que explicam a maior parte do resultado financeiro.

Do ponto de vista executivo, a empresa deve priorizar ações em Informática e Periféricos, sem deixar de acompanhar produtos e vendedores individualmente.

In [ ]:
# FUNÇÃO DE PARETO
# ============================================================

def analise_pareto(base, dimensao, metrica):
    tabela = base.groupby(dimensao).agg(
        valor_total=(metrica, "sum"),
        quantidade_registros=("id_venda", "count")
    ).reset_index()

    tabela = tabela.sort_values("valor_total", ascending=False).reset_index(drop=True)

    tabela.insert(0, "ranking", range(1, len(tabela) + 1))

    total = tabela["valor_total"].sum()

    tabela["participacao_%"] = tabela["valor_total"] / total * 100
    tabela["participacao_acumulada_%"] = tabela["participacao_%"].cumsum()

    indice_80 = tabela[tabela["participacao_acumulada_%"] >= 80].index.min()

    tabela["grupo_pareto_80"] = np.where(
        tabela.index <= indice_80,
        "Sim",
        "Não"
    )

    return tabela

In [ ]:
# PARETOS PRINCIPAIS
# ============================================================

pareto_produto_lucro = analise_pareto(df, "produto", "lucro")
pareto_categoria_lucro = analise_pareto(df, "categoria", "lucro")
pareto_regiao_lucro = analise_pareto(df, "regiao", "lucro")
pareto_vendedor_lucro = analise_pareto(df, "vendedor", "lucro")
pareto_marca_lucro = analise_pareto(df, "marca", "lucro")
pareto_linha_lucro = analise_pareto(df, "linha_produto", "lucro")
pareto_cliente_receita = analise_pareto(df, "cliente", "receita")
pareto_cliente_lucro = analise_pareto(df, "cliente", "lucro")

In [ ]:
# RESUMO EXECUTIVO DOS PARETOS
# ============================================================

analises_pareto = {
    "Produto vs. Lucro": pareto_produto_lucro,
    "Categoria vs. Lucro": pareto_categoria_lucro,
    "Região vs. Lucro": pareto_regiao_lucro,
    "Vendedor vs. Lucro": pareto_vendedor_lucro,
    "Marca vs. Lucro": pareto_marca_lucro,
    "Linha vs. Lucro": pareto_linha_lucro,
    "Cliente vs. Receita": pareto_cliente_receita,
    "Cliente vs. Lucro": pareto_cliente_lucro
}

resumo_pareto = []

for nome, tabela in analises_pareto.items():
    grupo_80 = tabela[tabela["grupo_pareto_80"] == "Sim"]

    resumo_pareto.append({
        "Análise": nome,
        "Total de itens": tabela.shape[0],
        "Itens até atingir 80%": grupo_80.shape[0],
        "% de itens necessários": grupo_80.shape[0] / tabela.shape[0],
        "Participação acumulada": grupo_80["participacao_acumulada_%"].max() / 100
    })

resumo_pareto = pd.DataFrame(resumo_pareto)

resumo_pareto

,Análise,Total de itens,Itens até atingir 80%,% de itens necessários,Participação acumulada
0,Produto vs. Lucro,8,6,0.75,0.80
1,Categoria vs. Lucro,4,2,0.50,0.84
2,Região vs. Lucro,5,4,0.80,0.83
3,Vendedor vs. Lucro,20,15,0.75,0.83
4,Marca vs. Lucro,4,4,1.00,1.00
5,Linha vs. Lucro,3,3,1.00,1.00
6,Cliente vs. Receita,199,118,0.59,0.80
7,Cliente vs. Lucro,199,102,0.51,0.80


### Análise Lucro Negativo
Foram identificadas vendas com lucro negativo. Isso significa que parte da receita da empresa está sendo gerada com prejuízo.

Esse é um ponto crítico porque mostra que faturamento alto não garante bom desempenho financeiro. A empresa precisa monitorar lucro e margem antes de aprovar vendas, principalmente quando há descontos elevados.

In [ ]:
# ANÁLISE DE VENDAS COM LUCRO NEGATIVO
# ============================================================

vendas_lucro_negativo = df[df["lucro"] < 0]

qtd_lucro_negativo = vendas_lucro_negativo.shape[0]
percentual_lucro_negativo = qtd_lucro_negativo / df.shape[0]
receita_lucro_negativo = vendas_lucro_negativo["receita"].sum()
prejuizo_total = vendas_lucro_negativo["lucro"].sum()

analise_lucro_negativo = pd.DataFrame({
    "Indicador": [
        "Quantidade de vendas com lucro negativo",
        "Percentual da base",
        "Receita gerada por essas vendas",
        "Resultado financeiro dessas vendas"
    ],
    "Valor": [
        qtd_lucro_negativo,
        formatar_percentual(percentual_lucro_negativo),
        formatar_moeda(receita_lucro_negativo),
        formatar_moeda(prejuizo_total)
    ]
})

analise_lucro_negativo

,Indicador,Valor
0,Quantidade de vendas com lucro negativo,123
1,Percentual da base,"12,30%"
2,Receita gerada por essas vendas,"R$ 1.590.355,85"
3,Resultado financeiro dessas vendas,"R$ -100.838,39"


In [ ]:
# FUNÇÕES PARA EXIBIÇÃO EXECUTIVA EM CARDS
# ============================================================

from IPython.display import display, Markdown

def icone_prioridade(prioridade):
    if prioridade == "Alta":
        return "🔴 Alta"
    elif prioridade == "Média":
        return "🟡 Média"
    elif prioridade == "Baixa":
        return "🟢 Baixa"
    else:
        return prioridade


def exibir_cards(df, titulo, campo_titulo, campos, numeracao=True):
    """
    Exibe um DataFrame em formato de cards Markdown,
    deixando informações textuais mais legíveis no notebook.

    Parâmetros:
    - df: DataFrame a ser exibido.
    - titulo: título geral da seção.
    - campo_titulo: coluna usada como título de cada card.
    - campos: lista de colunas a serem exibidas dentro de cada card.
    - numeracao: se True, numera os cards.
    """

    texto = f"## {titulo}\n\n"

    for i, (_, linha) in enumerate(df.iterrows(), start=1):
        titulo_card = linha[campo_titulo]

        if numeracao:
            texto += f"### {i}. {titulo_card}\n\n"
        else:
            texto += f"### {titulo_card}\n\n"

        for campo in campos:
            valor = linha[campo]

            if campo.lower() == "prioridade":
                valor = icone_prioridade(valor)

            texto += f"**{campo}:** {valor}\n\n"

        texto += "---\n\n"

    display(Markdown(texto))

In [ ]:
# 11. SÍNTESE DOS PRINCIPAIS INSIGHTS
# ============================================================

insights_executivos = pd.DataFrame({
    "Insight": [
        "A empresa gera receita e lucro, mas possui baixa taxa de atingimento de metas",
        "Descontos elevados reduzem fortemente a margem de lucro",
        "Existem vendas com lucro negativo",
        "A região Sul é a principal referência de desempenho",
        "O Centro-Oeste vende bem, mas possui baixa margem",
        "O canal Online tem maior escala, enquanto a Loja tem margem superior",
        "Produtos com maior receita não são necessariamente os mais rentáveis",
        "Clientes recorrentes sustentam volume, mas clientes novos têm boa margem",
        "Atrasos de entrega são um problema operacional relevante",
        "Categorias Informática e Periféricos concentram a maior parte do lucro"
    ],
    "Evidência": [
        "A empresa apresenta receita e lucro relevantes, porém apenas cerca de 7% das vendas atingiram a meta individual.",
        "A análise mostra relação negativa entre desconto e margem. Vendas com descontos mais altos apresentam margens menores.",
        "Foram identificadas vendas em que o lucro final foi menor que zero, ou seja, vendas que geraram prejuízo.",
        "A região Sul apresentou uma das melhores combinações de receita, lucro e margem de lucro.",
        "O Centro-Oeste possui alta receita, mas apresenta a menor margem entre as regiões analisadas.",
        "O canal Online lidera em receita e lucro absoluto, enquanto a Loja apresenta margem de lucro ligeiramente superior.",
        "O Monitor lidera em receita, mas produtos como Notebook e Impressora apresentam margens superiores.",
        "Clientes recorrentes geram maior receita e lucro absoluto, enquanto clientes novos apresentam margem ligeiramente superior.",
        "Mais da metade das entregas foram classificadas como atrasadas.",
        "A análise de Pareto mostrou que Informática e Periféricos concentram mais de 80% do lucro total."
    ],
    "Interpretação Executiva": [
        "As metas podem estar mal calibradas ou distribuídas de forma inadequada entre produtos, regiões, canais e vendedores.",
        "A empresa pode estar sacrificando rentabilidade para aumentar o volume de vendas.",
        "Parte do faturamento está destruindo valor financeiro, pois gera receita, mas reduz o lucro final.",
        "A região pode servir como referência para boas práticas comerciais e operacionais.",
        "A região precisa ser investigada para entender se o problema vem de descontos, custos, mix de produtos ou atuação comercial.",
        "Cada canal possui um papel estratégico diferente: o Online gera escala, enquanto a Loja apresenta melhor eficiência proporcional.",
        "A empresa não deve tomar decisões apenas por faturamento, mas também por margem e lucro.",
        "A empresa deve equilibrar estratégias de fidelização e aquisição, respeitando a rentabilidade de cada perfil.",
        "A operação logística deve ser tratada como fator estratégico, pois pode afetar satisfação, recompra e desempenho comercial.",
        "Essas categorias devem ser protegidas e priorizadas em campanhas, estoque e análise gerencial."
    ],
    "Decisão Recomendada": [
        "Revisar as metas considerando região, canal, produto, vendedor e histórico de desempenho.",
        "Criar uma política formal de descontos com níveis de aprovação.",
        "Criar alerta ou bloqueio para vendas com margem negativa.",
        "Investigar e replicar as práticas da região Sul em outras regiões.",
        "Auditar descontos, custos, produtos e vendedores do Centro-Oeste.",
        "Investir no Online para escala e melhorar os processos operacionais da Loja.",
        "Priorizar produtos com maior margem em campanhas e revisar produtos com alta receita e baixa rentabilidade.",
        "Criar estratégias específicas para clientes recorrentes e novos.",
        "Monitorar atraso por região, canal, produto e vendedor.",
        "Priorizar Informática e Periféricos na estratégia comercial."
    ],
    "Prioridade": [
        "Alta",
        "Alta",
        "Alta",
        "Média",
        "Média",
        "Média",
        "Média",
        "Média",
        "Alta",
        "Alta"
    ]
})

In [ ]:
exibir_cards(
    df=insights_executivos,
    titulo="Síntese Executiva dos Principais Insights",
    campo_titulo="Insight",
    campos=[
        "Evidência",
        "Interpretação Executiva",
        "Decisão Recomendada",
        "Prioridade"
    ]
)

## Síntese Executiva dos Principais Insights

### 1. A empresa gera receita e lucro, mas possui baixa taxa de atingimento de metas

**Evidência:** A empresa apresenta receita e lucro relevantes, porém apenas cerca de 7% das vendas atingiram a meta individual.

**Interpretação Executiva:** As metas podem estar mal calibradas ou distribuídas de forma inadequada entre produtos, regiões, canais e vendedores.

**Decisão Recomendada:** Revisar as metas considerando região, canal, produto, vendedor e histórico de desempenho.

**Prioridade:** 🔴 Alta

---

### 2. Descontos elevados reduzem fortemente a margem de lucro

**Evidência:** A análise mostra relação negativa entre desconto e margem. Vendas com descontos mais altos apresentam margens menores.

**Interpretação Executiva:** A empresa pode estar sacrificando rentabilidade para aumentar o volume de vendas.

**Decisão Recomendada:** Criar uma política formal de descontos com níveis de aprovação.

**Prioridade:** 🔴 Alta

---

### 3. Existem vendas com lucro negativo

**Evidência:** Foram identificadas vendas em que o lucro final foi menor que zero, ou seja, vendas que geraram prejuízo.

**Interpretação Executiva:** Parte do faturamento está destruindo valor financeiro, pois gera receita, mas reduz o lucro final.

**Decisão Recomendada:** Criar alerta ou bloqueio para vendas com margem negativa.

**Prioridade:** 🔴 Alta

---

### 4. A região Sul é a principal referência de desempenho

**Evidência:** A região Sul apresentou uma das melhores combinações de receita, lucro e margem de lucro.

**Interpretação Executiva:** A região pode servir como referência para boas práticas comerciais e operacionais.

**Decisão Recomendada:** Investigar e replicar as práticas da região Sul em outras regiões.

**Prioridade:** 🟡 Média

---

### 5. O Centro-Oeste vende bem, mas possui baixa margem

**Evidência:** O Centro-Oeste possui alta receita, mas apresenta a menor margem entre as regiões analisadas.

**Interpretação Executiva:** A região precisa ser investigada para entender se o problema vem de descontos, custos, mix de produtos ou atuação comercial.

**Decisão Recomendada:** Auditar descontos, custos, produtos e vendedores do Centro-Oeste.

**Prioridade:** 🟡 Média

---

### 6. O canal Online tem maior escala, enquanto a Loja tem margem superior

**Evidência:** O canal Online lidera em receita e lucro absoluto, enquanto a Loja apresenta margem de lucro ligeiramente superior.

**Interpretação Executiva:** Cada canal possui um papel estratégico diferente: o Online gera escala, enquanto a Loja apresenta melhor eficiência proporcional.

**Decisão Recomendada:** Investir no Online para escala e melhorar os processos operacionais da Loja.

**Prioridade:** 🟡 Média

---

### 7. Produtos com maior receita não são necessariamente os mais rentáveis

**Evidência:** O Monitor lidera em receita, mas produtos como Notebook e Impressora apresentam margens superiores.

**Interpretação Executiva:** A empresa não deve tomar decisões apenas por faturamento, mas também por margem e lucro.

**Decisão Recomendada:** Priorizar produtos com maior margem em campanhas e revisar produtos com alta receita e baixa rentabilidade.

**Prioridade:** 🟡 Média

---

### 8. Clientes recorrentes sustentam volume, mas clientes novos têm boa margem

**Evidência:** Clientes recorrentes geram maior receita e lucro absoluto, enquanto clientes novos apresentam margem ligeiramente superior.

**Interpretação Executiva:** A empresa deve equilibrar estratégias de fidelização e aquisição, respeitando a rentabilidade de cada perfil.

**Decisão Recomendada:** Criar estratégias específicas para clientes recorrentes e novos.

**Prioridade:** 🟡 Média

---

### 9. Atrasos de entrega são um problema operacional relevante

**Evidência:** Mais da metade das entregas foram classificadas como atrasadas.

**Interpretação Executiva:** A operação logística deve ser tratada como fator estratégico, pois pode afetar satisfação, recompra e desempenho comercial.

**Decisão Recomendada:** Monitorar atraso por região, canal, produto e vendedor.

**Prioridade:** 🔴 Alta

---

### 10. Categorias Informática e Periféricos concentram a maior parte do lucro

**Evidência:** A análise de Pareto mostrou que Informática e Periféricos concentram mais de 80% do lucro total.

**Interpretação Executiva:** Essas categorias devem ser protegidas e priorizadas em campanhas, estoque e análise gerencial.

**Decisão Recomendada:** Priorizar Informática e Periféricos na estratégia comercial.

**Prioridade:** 🔴 Alta

---



In [ ]:
# 12. RESPOSTAS ÀS PERGUNTAS DE NEGÓCIO
# ============================================================

respostas_perguntas_negocio = pd.DataFrame({
    "Pergunta de Negócio": [
        "Quais regiões geram maior receita e quais apresentam menor margem de lucro?",
        "Quais produtos e categorias são mais vendidos e quais geram maior lucro?",
        "O canal online apresenta desempenho melhor ou pior que a loja?",
        "Quais vendedores possuem melhor desempenho?",
        "Clientes recorrentes geram maior receita e lucro do que clientes novos?",
        "O nível de desconto impacta negativamente a margem de lucro?",
        "Existe relação entre atraso na entrega e pior desempenho comercial?",
        "Quais marcas ou linhas de produto apresentam maior contribuição para o lucro?"
    ],
    "Resposta Executiva": [
        "A região Sul apresenta a melhor combinação de receita, lucro e margem. O Centro-Oeste possui alta receita, mas apresenta menor margem.",
        "Informática e Periféricos concentram a maior parte do lucro. Em produtos, Monitor lidera em receita, enquanto Notebook e Impressora apresentam melhores margens.",
        "O Online tem maior receita e maior lucro absoluto. A Loja tem margem ligeiramente superior, mas apresenta maior taxa de atraso.",
        "O Vendedor_20 se destaca em lucro e margem, enquanto o Vendedor_13 lidera em receita.",
        "Sim. Clientes recorrentes geram maior receita e lucro absoluto, mas clientes novos apresentam margem ligeiramente superior.",
        "Sim. A relação entre desconto e margem é negativa. Descontos maiores estão associados a margens menores.",
        "Há indícios. As vendas entregues no prazo apresentam maior taxa de atingimento de meta do que as vendas atrasadas.",
        "A marca Alpha lidera em lucro. As linhas Econômico, Premium e Padrão contribuem de forma relativamente equilibrada."
    ],
    "Decisão Recomendada": [
        "Replicar boas práticas do Sul e investigar a baixa margem do Centro-Oeste.",
        "Priorizar categorias e produtos por lucro e margem, não apenas por receita.",
        "Usar o Online para escala e corrigir gargalos operacionais da Loja.",
        "Avaliar vendedores por lucro, margem, meta e desconto médio.",
        "Manter fidelização de recorrentes e captar novos clientes com controle de desconto.",
        "Criar política formal de descontos e limites de aprovação.",
        "Monitorar atrasos como indicador estratégico.",
        "Fortalecer marcas e linhas com maior contribuição financeira."
    ]
})

In [ ]:
exibir_cards(
    df=respostas_perguntas_negocio,
    titulo="Respostas às Perguntas de Negócio",
    campo_titulo="Pergunta de Negócio",
    campos=[
        "Resposta Executiva",
        "Decisão Recomendada"
    ]
)

## Respostas às Perguntas de Negócio

### 1. Quais regiões geram maior receita e quais apresentam menor margem de lucro?

**Resposta Executiva:** A região Sul apresenta a melhor combinação de receita, lucro e margem. O Centro-Oeste possui alta receita, mas apresenta menor margem.

**Decisão Recomendada:** Replicar boas práticas do Sul e investigar a baixa margem do Centro-Oeste.

---

### 2. Quais produtos e categorias são mais vendidos e quais geram maior lucro?

**Resposta Executiva:** Informática e Periféricos concentram a maior parte do lucro. Em produtos, Monitor lidera em receita, enquanto Notebook e Impressora apresentam melhores margens.

**Decisão Recomendada:** Priorizar categorias e produtos por lucro e margem, não apenas por receita.

---

### 3. O canal online apresenta desempenho melhor ou pior que a loja?

**Resposta Executiva:** O Online tem maior receita e maior lucro absoluto. A Loja tem margem ligeiramente superior, mas apresenta maior taxa de atraso.

**Decisão Recomendada:** Usar o Online para escala e corrigir gargalos operacionais da Loja.

---

### 4. Quais vendedores possuem melhor desempenho?

**Resposta Executiva:** O Vendedor_20 se destaca em lucro e margem, enquanto o Vendedor_13 lidera em receita.

**Decisão Recomendada:** Avaliar vendedores por lucro, margem, meta e desconto médio.

---

### 5. Clientes recorrentes geram maior receita e lucro do que clientes novos?

**Resposta Executiva:** Sim. Clientes recorrentes geram maior receita e lucro absoluto, mas clientes novos apresentam margem ligeiramente superior.

**Decisão Recomendada:** Manter fidelização de recorrentes e captar novos clientes com controle de desconto.

---

### 6. O nível de desconto impacta negativamente a margem de lucro?

**Resposta Executiva:** Sim. A relação entre desconto e margem é negativa. Descontos maiores estão associados a margens menores.

**Decisão Recomendada:** Criar política formal de descontos e limites de aprovação.

---

### 7. Existe relação entre atraso na entrega e pior desempenho comercial?

**Resposta Executiva:** Há indícios. As vendas entregues no prazo apresentam maior taxa de atingimento de meta do que as vendas atrasadas.

**Decisão Recomendada:** Monitorar atrasos como indicador estratégico.

---

### 8. Quais marcas ou linhas de produto apresentam maior contribuição para o lucro?

**Resposta Executiva:** A marca Alpha lidera em lucro. As linhas Econômico, Premium e Padrão contribuem de forma relativamente equilibrada.

**Decisão Recomendada:** Fortalecer marcas e linhas com maior contribuição financeira.

---



In [ ]:
# 13. MATRIZ DE DECISÃO ESTRATÉGICA
# ============================================================

matriz_decisao = pd.DataFrame({
    "Problema Identificado": [
        "Baixo atingimento de metas",
        "Descontos elevados reduzem margem",
        "Vendas com lucro negativo",
        "Alta taxa de entregas atrasadas",
        "Centro-Oeste com baixa margem",
        "Produtos de alta receita com margem menor",
        "Avaliação comercial focada apenas em receita",
        "Dependência forte de duas categorias no lucro"
    ],
    "Evidência nos Dados": [
        "Apenas cerca de 7% das vendas atingiram a meta individual.",
        "Descontos maiores apresentam relação negativa com a margem de lucro.",
        "Existem vendas em que o lucro final é menor que zero.",
        "Mais de 50% das entregas estão atrasadas.",
        "A região possui alta receita, mas menor margem entre as regiões.",
        "Monitor lidera em receita, mas não lidera em margem.",
        "O vendedor líder em receita não é necessariamente o líder em lucro.",
        "Informática e Periféricos concentram mais de 80% do lucro."
    ],
    "Impacto no Negócio": [
        "Dificulta a avaliação real da performance comercial.",
        "Reduz a rentabilidade mesmo quando o volume de vendas aumenta.",
        "Gera faturamento com destruição de valor financeiro.",
        "Pode afetar satisfação, recompra e percepção de qualidade.",
        "Gera alto faturamento com baixa eficiência financeira.",
        "Pode levar a decisões comerciais que aumentam receita, mas reduzem lucro.",
        "Pode incentivar vendedores a priorizarem faturamento em vez de rentabilidade.",
        "Cria concentração estratégica em poucas categorias."
    ],
    "Recomendação Estratégica": [
        "Recalibrar metas por região, canal, vendedor e produto.",
        "Criar política formal de descontos com níveis de aprovação.",
        "Criar alerta ou bloqueio para vendas com margem negativa.",
        "Monitorar atraso por canal, região, produto e vendedor.",
        "Auditar descontos, custos, produtos e vendedores da região.",
        "Revisar preço, desconto e fornecedores de produtos com baixa margem.",
        "Criar ranking comercial baseado em lucro, margem e meta.",
        "Proteger categorias-chave e diversificar oportunidades."
    ],
    "Prioridade": [
        "Alta",
        "Alta",
        "Alta",
        "Alta",
        "Média",
        "Média",
        "Média",
        "Alta"
    ]
})

In [ ]:
# Separar por prioridade para facilitar leitura
matriz_alta = matriz_decisao[matriz_decisao["Prioridade"] == "Alta"]
matriz_media = matriz_decisao[matriz_decisao["Prioridade"] == "Média"]

exibir_cards(
    df=matriz_alta,
    titulo="Matriz de Decisão Estratégica — Prioridade Alta",
    campo_titulo="Problema Identificado",
    campos=[
        "Evidência nos Dados",
        "Impacto no Negócio",
        "Recomendação Estratégica",
        "Prioridade"
    ]
)

exibir_cards(
    df=matriz_media,
    titulo="Matriz de Decisão Estratégica — Prioridade Média",
    campo_titulo="Problema Identificado",
    campos=[
        "Evidência nos Dados",
        "Impacto no Negócio",
        "Recomendação Estratégica",
        "Prioridade"
    ]
)

## Matriz de Decisão Estratégica — Prioridade Alta

### 1. Baixo atingimento de metas

**Evidência nos Dados:** Apenas cerca de 7% das vendas atingiram a meta individual.

**Impacto no Negócio:** Dificulta a avaliação real da performance comercial.

**Recomendação Estratégica:** Recalibrar metas por região, canal, vendedor e produto.

**Prioridade:** 🔴 Alta

---

### 2. Descontos elevados reduzem margem

**Evidência nos Dados:** Descontos maiores apresentam relação negativa com a margem de lucro.

**Impacto no Negócio:** Reduz a rentabilidade mesmo quando o volume de vendas aumenta.

**Recomendação Estratégica:** Criar política formal de descontos com níveis de aprovação.

**Prioridade:** 🔴 Alta

---

### 3. Vendas com lucro negativo

**Evidência nos Dados:** Existem vendas em que o lucro final é menor que zero.

**Impacto no Negócio:** Gera faturamento com destruição de valor financeiro.

**Recomendação Estratégica:** Criar alerta ou bloqueio para vendas com margem negativa.

**Prioridade:** 🔴 Alta

---

### 4. Alta taxa de entregas atrasadas

**Evidência nos Dados:** Mais de 50% das entregas estão atrasadas.

**Impacto no Negócio:** Pode afetar satisfação, recompra e percepção de qualidade.

**Recomendação Estratégica:** Monitorar atraso por canal, região, produto e vendedor.

**Prioridade:** 🔴 Alta

---

### 5. Dependência forte de duas categorias no lucro

**Evidência nos Dados:** Informática e Periféricos concentram mais de 80% do lucro.

**Impacto no Negócio:** Cria concentração estratégica em poucas categorias.

**Recomendação Estratégica:** Proteger categorias-chave e diversificar oportunidades.

**Prioridade:** 🔴 Alta

---



## Matriz de Decisão Estratégica — Prioridade Média

### 1. Centro-Oeste com baixa margem

**Evidência nos Dados:** A região possui alta receita, mas menor margem entre as regiões.

**Impacto no Negócio:** Gera alto faturamento com baixa eficiência financeira.

**Recomendação Estratégica:** Auditar descontos, custos, produtos e vendedores da região.

**Prioridade:** 🟡 Média

---

### 2. Produtos de alta receita com margem menor

**Evidência nos Dados:** Monitor lidera em receita, mas não lidera em margem.

**Impacto no Negócio:** Pode levar a decisões comerciais que aumentam receita, mas reduzem lucro.

**Recomendação Estratégica:** Revisar preço, desconto e fornecedores de produtos com baixa margem.

**Prioridade:** 🟡 Média

---

### 3. Avaliação comercial focada apenas em receita

**Evidência nos Dados:** O vendedor líder em receita não é necessariamente o líder em lucro.

**Impacto no Negócio:** Pode incentivar vendedores a priorizarem faturamento em vez de rentabilidade.

**Recomendação Estratégica:** Criar ranking comercial baseado em lucro, margem e meta.

**Prioridade:** 🟡 Média

---



In [ ]:
# 14. PLANO DE AÇÃO EXECUTIVO
# ============================================================

plano_acao = pd.DataFrame({
    "Prazo": [
        "Curto prazo",
        "Curto prazo",
        "Curto prazo",
        "Curto prazo",
        "Médio prazo",
        "Médio prazo",
        "Médio prazo",
        "Longo prazo",
        "Longo prazo"
    ],
    "Ação": [
        "Criar alerta para vendas com lucro negativo",
        "Bloquear ou exigir aprovação para descontos acima de 20%",
        "Monitorar taxa de atraso por canal e região",
        "Identificar vendedores com maior desconto médio",
        "Recalibrar metas por produto, região, canal e vendedor",
        "Revisar mix de produtos do Centro-Oeste",
        "Criar ranking comercial por lucro, margem e meta",
        "Implantar dashboard recorrente de BI",
        "Automatizar acompanhamento mensal dos KPIs estratégicos"
    ],
    "Objetivo": [
        "Evitar vendas que geram prejuízo.",
        "Reduzir perda de margem em vendas altamente descontadas.",
        "Identificar gargalos operacionais prioritários.",
        "Corrigir práticas comerciais pouco rentáveis.",
        "Tornar as metas mais realistas e aderentes ao histórico.",
        "Melhorar a rentabilidade regional.",
        "Avaliar a equipe comercial com mais precisão.",
        "Acompanhar resultados de forma visual e recorrente.",
        "Transformar BI em rotina de gestão."
    ],
    "Área Responsável": [
        "Comercial / Financeiro",
        "Comercial / Gerência",
        "Operações / Logística",
        "Comercial",
        "Gestão Comercial",
        "Comercial / Produtos",
        "Gestão Comercial",
        "BI / Gestão",
        "BI / Gestão"
    ],
    "Prioridade": [
        "Alta",
        "Alta",
        "Alta",
        "Média",
        "Alta",
        "Média",
        "Média",
        "Alta",
        "Média"
    ]
})

In [ ]:
for prazo in ["Curto prazo", "Médio prazo", "Longo prazo"]:
    plano_prazo = plano_acao[plano_acao["Prazo"] == prazo]

    exibir_cards(
        df=plano_prazo,
        titulo=f"Plano de Ação — {prazo}",
        campo_titulo="Ação",
        campos=[
            "Objetivo",
            "Área Responsável",
            "Prioridade"
        ]
    )

## Plano de Ação — Curto prazo

### 1. Criar alerta para vendas com lucro negativo

**Objetivo:** Evitar vendas que geram prejuízo.

**Área Responsável:** Comercial / Financeiro

**Prioridade:** 🔴 Alta

---

### 2. Bloquear ou exigir aprovação para descontos acima de 20%

**Objetivo:** Reduzir perda de margem em vendas altamente descontadas.

**Área Responsável:** Comercial / Gerência

**Prioridade:** 🔴 Alta

---

### 3. Monitorar taxa de atraso por canal e região

**Objetivo:** Identificar gargalos operacionais prioritários.

**Área Responsável:** Operações / Logística

**Prioridade:** 🔴 Alta

---

### 4. Identificar vendedores com maior desconto médio

**Objetivo:** Corrigir práticas comerciais pouco rentáveis.

**Área Responsável:** Comercial

**Prioridade:** 🟡 Média

---



## Plano de Ação — Médio prazo

### 1. Recalibrar metas por produto, região, canal e vendedor

**Objetivo:** Tornar as metas mais realistas e aderentes ao histórico.

**Área Responsável:** Gestão Comercial

**Prioridade:** 🔴 Alta

---

### 2. Revisar mix de produtos do Centro-Oeste

**Objetivo:** Melhorar a rentabilidade regional.

**Área Responsável:** Comercial / Produtos

**Prioridade:** 🟡 Média

---

### 3. Criar ranking comercial por lucro, margem e meta

**Objetivo:** Avaliar a equipe comercial com mais precisão.

**Área Responsável:** Gestão Comercial

**Prioridade:** 🟡 Média

---



## Plano de Ação — Longo prazo

### 1. Implantar dashboard recorrente de BI

**Objetivo:** Acompanhar resultados de forma visual e recorrente.

**Área Responsável:** BI / Gestão

**Prioridade:** 🔴 Alta

---

### 2. Automatizar acompanhamento mensal dos KPIs estratégicos

**Objetivo:** Transformar BI em rotina de gestão.

**Área Responsável:** BI / Gestão

**Prioridade:** 🟡 Média

---



In [ ]:
# EXPORTAÇÃO DOS RESULTADOS DA ETAPA 7
# ============================================================

pasta_saida = Path("outputs_etapa7")
pasta_saida.mkdir(exist_ok=True)

kpis_finais.to_csv(pasta_saida / "kpis_finais.csv", index=False, encoding="utf-8-sig")
analise_regiao.to_csv(pasta_saida / "analise_regiao.csv", index=False, encoding="utf-8-sig")
analise_produto.to_csv(pasta_saida / "analise_produto.csv", index=False, encoding="utf-8-sig")
analise_categoria.to_csv(pasta_saida / "analise_categoria.csv", index=False, encoding="utf-8-sig")
analise_canal.to_csv(pasta_saida / "analise_canal.csv", index=False, encoding="utf-8-sig")
analise_vendedor.to_csv(pasta_saida / "analise_vendedor.csv", index=False, encoding="utf-8-sig")
analise_tipo_cliente.to_csv(pasta_saida / "analise_tipo_cliente.csv", index=False, encoding="utf-8-sig")
analise_desconto.to_csv(pasta_saida / "analise_desconto.csv", index=False, encoding="utf-8-sig")
analise_entrega.to_csv(pasta_saida / "analise_entrega.csv", index=False, encoding="utf-8-sig")
analise_marca.to_csv(pasta_saida / "analise_marca.csv", index=False, encoding="utf-8-sig")
analise_linha.to_csv(pasta_saida / "analise_linha_produto.csv", index=False, encoding="utf-8-sig")
resumo_pareto.to_csv(pasta_saida / "resumo_pareto.csv", index=False, encoding="utf-8-sig")
insights_executivos.to_csv(pasta_saida / "insights_executivos.csv", index=False, encoding="utf-8-sig")
respostas_perguntas_negocio.to_csv(pasta_saida / "respostas_perguntas_negocio.csv", index=False, encoding="utf-8-sig")
matriz_decisao.to_csv(pasta_saida / "matriz_decisao.csv", index=False, encoding="utf-8-sig")
plano_acao.to_csv(pasta_saida / "plano_acao.csv", index=False, encoding="utf-8-sig")

print("Resultados da Etapa 7 exportados com sucesso.")

Resultados da Etapa 7 exportados com sucesso.


## Etapa 7 - Insights e Tomada de Decisão

Nesta etapa foram consolidados os principais insights obtidos ao longo do projeto, com o objetivo de transformar as análises realizadas em recomendações estratégicas para a empresa Comercial Insight Ltda.

A análise final mostrou que a empresa possui uma operação lucrativa, com receita total aproximada de R$ 15,20 milhões e lucro total de aproximadamente R$ 3,08 milhões. A margem geral ficou em torno de 20,28%, indicando que a empresa consegue transformar parte relevante da receita em lucro.

Apesar disso, foram identificados problemas importantes. A taxa de vendas que atingiram a meta foi de apenas 7%, o que indica possível desalinhamento na definição das metas comerciais. Além disso, a taxa de entregas atrasadas foi superior a 50%, mostrando que a operação logística deve ser tratada como uma dimensão estratégica do negócio.

Um dos principais insights está relacionado à política de descontos. A análise demonstrou que descontos maiores estão associados a margens menores. Essa relação foi confirmada tanto pela correlação entre desconto e margem de lucro quanto pela análise por faixas de desconto. Dessa forma, recomenda-se criar uma política formal de descontos, com níveis de aprovação para faixas mais altas.

Também foram identificadas vendas com lucro negativo. Esse resultado mostra que algumas transações geram faturamento, mas reduzem o resultado financeiro da empresa. Recomenda-se criar alertas ou bloqueios para vendas com margem negativa, exigindo aprovação gerencial antes da conclusão.

Na análise regional, a região Sul se destacou como referência de desempenho, apresentando alta receita, alto lucro e melhor margem. Por outro lado, o Centro-Oeste apresentou alta receita, mas baixa margem, indicando necessidade de investigação sobre descontos, custos, mix de produtos e atuação comercial.

A análise por canal mostrou que o canal Online possui maior escala, com maior receita e lucro absoluto, enquanto a Loja possui margem ligeiramente superior. No entanto, a loja também apresentou maior taxa de atraso, o que exige atenção operacional.

Na análise de produtos e categorias, observou-se que produtos com maior receita não necessariamente são os mais rentáveis. O Monitor liderou em receita, mas produtos como Notebook e Impressora apresentaram margens superiores. A análise de Pareto mostrou ainda que as categorias Informática e Periféricos concentram mais de 80% do lucro total, devendo ser tratadas como categorias estratégicas.

A análise dos vendedores mostrou que o desempenho comercial deve ser avaliado de forma multidimensional. O vendedor com maior receita não necessariamente é o mesmo com maior lucro ou margem. Por isso, recomenda-se avaliar a equipe com base em receita, lucro, margem, taxa de meta e desconto médio.

Clientes recorrentes apresentaram maior receita e lucro absoluto, reforçando a importância da fidelização. Já clientes novos apresentaram margem ligeiramente superior, indicando potencial de rentabilidade desde que a política de descontos seja controlada.

A matriz de decisão estratégica consolidou os principais problemas identificados, suas evidências, impactos e recomendações. Entre as ações prioritárias estão: revisar metas comerciais, criar política formal de descontos, controlar vendas com lucro negativo, monitorar atrasos de entrega e priorizar categorias com maior contribuição para o lucro.

Conclui-se que a Comercial Insight Ltda. deve deixar de avaliar desempenho apenas por faturamento e passar a tomar decisões com base em lucro, margem, metas, descontos, operação e comportamento dos clientes. Essa mudança permitirá decisões mais sustentáveis e orientadas por evidências.

In [ ]:
# 15. DIRECIONAMENTO PARA A ETAPA 6 - VISUALIZAÇÕES
# ============================================================

direcionamento_etapa6 = pd.DataFrame({
    "Insight da Etapa 7": [
        "Baixo atingimento de metas",
        "Descontos reduzem margem",
        "Região Sul tem melhor desempenho",
        "Centro-Oeste tem baixa margem",
        "Online vende mais; Loja tem margem superior",
        "Vendedores variam em lucro, margem e meta",
        "Informática e Periféricos concentram lucro",
        "Atrasos são problema operacional",
        "Produtos com maior receita não são sempre os mais rentáveis"
    ],
    "Visualização Recomendada": [
        "Card de KPI com taxa de atingimento de meta",
        "Gráfico de margem por faixa de desconto",
        "Gráfico de receita e margem por região",
        "Gráfico comparativo de margem regional",
        "Gráfico de receita, lucro e margem por canal",
        "Ranking de vendedores por lucro e margem",
        "Gráfico de Pareto por categoria",
        "Gráfico de taxa de meta por status de entrega",
        "Comparativo de receita e margem por produto"
    ],
    "Objetivo da Visualização": [
        "Mostrar rapidamente o principal problema comercial.",
        "Evidenciar a perda de rentabilidade com descontos altos.",
        "Destacar a região que pode servir como referência de boas práticas.",
        "Expor a região que precisa de revisão estratégica.",
        "Comparar o papel estratégico de cada canal.",
        "Avaliar desempenho comercial de forma mais justa.",
        "Mostrar concentração do lucro por categoria.",
        "Relacionar operação e desempenho comercial.",
        "Evitar decisão baseada apenas em faturamento."
    ],
    "Tipo de Gráfico Sugerido": [
        "Card/KPI",
        "Barras",
        "Barras combinadas",
        "Barras",
        "Barras agrupadas",
        "Barras horizontais",
        "Pareto",
        "Barras",
        "Barras ou dispersão"
    ]
})

In [ ]:
exibir_cards(
    df=direcionamento_etapa6,
    titulo="Direcionamento para a Etapa 6 — Visualizações Recomendadas",
    campo_titulo="Insight da Etapa 7",
    campos=[
        "Visualização Recomendada",
        "Tipo de Gráfico Sugerido",
        "Objetivo da Visualização"
    ]
)

## Direcionamento para a Etapa 6 — Visualizações Recomendadas

### 1. Baixo atingimento de metas

**Visualização Recomendada:** Card de KPI com taxa de atingimento de meta

**Tipo de Gráfico Sugerido:** Card/KPI

**Objetivo da Visualização:** Mostrar rapidamente o principal problema comercial.

---

### 2. Descontos reduzem margem

**Visualização Recomendada:** Gráfico de margem por faixa de desconto

**Tipo de Gráfico Sugerido:** Barras

**Objetivo da Visualização:** Evidenciar a perda de rentabilidade com descontos altos.

---

### 3. Região Sul tem melhor desempenho

**Visualização Recomendada:** Gráfico de receita e margem por região

**Tipo de Gráfico Sugerido:** Barras combinadas

**Objetivo da Visualização:** Destacar a região que pode servir como referência de boas práticas.

---

### 4. Centro-Oeste tem baixa margem

**Visualização Recomendada:** Gráfico comparativo de margem regional

**Tipo de Gráfico Sugerido:** Barras

**Objetivo da Visualização:** Expor a região que precisa de revisão estratégica.

---

### 5. Online vende mais; Loja tem margem superior

**Visualização Recomendada:** Gráfico de receita, lucro e margem por canal

**Tipo de Gráfico Sugerido:** Barras agrupadas

**Objetivo da Visualização:** Comparar o papel estratégico de cada canal.

---

### 6. Vendedores variam em lucro, margem e meta

**Visualização Recomendada:** Ranking de vendedores por lucro e margem

**Tipo de Gráfico Sugerido:** Barras horizontais

**Objetivo da Visualização:** Avaliar desempenho comercial de forma mais justa.

---

### 7. Informática e Periféricos concentram lucro

**Visualização Recomendada:** Gráfico de Pareto por categoria

**Tipo de Gráfico Sugerido:** Pareto

**Objetivo da Visualização:** Mostrar concentração do lucro por categoria.

---

### 8. Atrasos são problema operacional

**Visualização Recomendada:** Gráfico de taxa de meta por status de entrega

**Tipo de Gráfico Sugerido:** Barras

**Objetivo da Visualização:** Relacionar operação e desempenho comercial.

---

### 9. Produtos com maior receita não são sempre os mais rentáveis

**Visualização Recomendada:** Comparativo de receita e margem por produto

**Tipo de Gráfico Sugerido:** Barras ou dispersão

**Objetivo da Visualização:** Evitar decisão baseada apenas em faturamento.

---

